In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

years = range(2019, 2025)

temp = (
    pd.read_csv("../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv")
      .replace({"#ERROR!": np.nan})
)

centralities = ["central", "peripheral"]
modes = ["fast", "medium", "slow"]

all_years_df = []

for year in years:
    print(f"Processing year {year}")

    port_dict = {}

    for centrality in centralities:
        # Option 1: If your files include n in the filename
        # port_dict[f"{centrality}"] = (
        #     pd.read_csv(
        #         f"../../data/07_portfolios_metadata/simple_{centrality}_metadata_{year}_n{n}.csv"
        #     )[["Ticker", "Country"]]
        #     .merge(temp, on="Ticker", how="inner")
        # )
        
        # Option 2: If files don't include n, just read once and track n separately
        port_dict[f"{centrality}"] = (
            pd.read_csv(
                f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}.csv"
            )[["Ticker", "Country"]]
            .merge(temp, on="Ticker", how="inner")
        )

    dfs = []
    for portfolio_name, df in port_dict.items():
        df = df.copy()
        df["portfolio"] = portfolio_name.replace("_", " ")
        df["year"] = year
        df["n"] = n  # Add n column to track parameter
        dfs.append(df)

    final_df = pd.concat(dfs, ignore_index=True)

    beta_df = pd.read_csv(
        f"../../data/07_portfolios_metadata/beta_df_{year}.csv"
    )
    if 'portfolio' in beta_df.columns:
        beta_df = beta_df.drop(columns=['portfolio'])
        
    momentum_df = pd.read_csv(
        f"../../data/07_portfolios_metadata/momentum_df_{year}.csv"
    )
    if 'portfolio' in momentum_df.columns:
        momentum_df = momentum_df.drop(columns=['portfolio'])

    final_df = final_df.merge(beta_df, on=["Ticker"], how="left")
    final_df = final_df.merge(momentum_df, on=["Ticker"], how="left")

    cols_to_clean = [
        col for col in final_df.columns
        if str(year) in col
    ]

    final_df[cols_to_clean] = (
        final_df[cols_to_clean]
            .apply(lambda s: s.astype(str).str.replace(",", ".", regex=False))
            .apply(pd.to_numeric, errors="coerce")
    )

    cols_to_keep = [
        "Ticker", "Country", "Company", "Sector",
        "Industry", "portfolio", "year", "n"  # Include n in columns to keep
    ] + cols_to_clean

    final_df = final_df[cols_to_keep]

    for col in cols_to_clean:
        final_df[f"{col}_cut"] = (
            pd.qcut(
                final_df[col],
                q=2,
                labels=False,
                duplicates="drop"
            ) + 1
        )

    all_years_df.append(final_df)

# dataframe final com todos os anos e diferentes valores de n
final_panel_df = pd.concat(all_years_df, ignore_index=True)

# Now you can filter by n value
# Example: df_n50 = final_panel_df[final_panel_df['n'] == 50]

Processing year 2019
Processing year 2020
Processing year 2021
Processing year 2022
Processing year 2023
Processing year 2024


In [32]:
final_panel_df

,Ticker,Country,Company,Sector,Industry,portfolio,year,n,mcap_2019,pe_2019,...,pscore_2024_cut,azscore_2024_cut,fcfyield_2024_cut,roic_2024_cut,rg_2024_cut,ndebitda_2024_cut,gpm_2024_cut,epsdg_2024_cut,beta_2024_cut,momentum12mo_2024_cut
0,ACGL,Bermuda,Arch Capital Group Ltd,Financial,Insurance - Diversified,central,2019,100,1.732293e+10,10.531762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACIW,USA,ACI Worldwide Inc,Technology,Software - Infrastructure,central,2019,100,4.394887e+09,65.630161,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ACWI,USA,iShares MSCI ACWI ETF,Financial,Exchange Traded Fund,central,2019,100,1.284242e+10,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACWX,USA,iShares MSCI ACWI ex US ETF,Financial,Exchange Traded Fund,central,2019,100,4.136346e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ADBE,USA,Adobe Inc,Technology,Software - Application,central,2019,100,1.589684e+11,50.999084,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,TOMZ,USA,TOMI Environmental Solutions Inc,Industrials,Pollution & Treatment Controls,peripheral,2024,100,NaN,NaN,...,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0
956,UONE,USA,Urban One Inc,Communication Services,Broadcasting,peripheral,2024,100,NaN,NaN,...,1.0,1.0,2.0,1.0,1.0,2.0,2.0,1.0,2.0,1.0
957,VIRC,USA,Virco Manufacturing Corp,Consumer Cyclical,"Furnishings, Fixtures & Appliances",peripheral,2024,100,NaN,NaN,...,1.0,NaN,2.0,2.0,2.0,1.0,1.0,2.0,2.0,1.0
958,WFCF,USA,Where Food Comes From Inc,Industrials,Specialty Business Services,peripheral,2024,100,NaN,NaN,...,1.0,2.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0


In [33]:
final_panel_df.query("portfolio=='central' and year==2019").shape

(80, 164)

In [34]:
final_panel_df.to_csv("../../data/07_portfolios_metadata/complete_metadata.csv")